In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv


load_dotenv()

llm=ChatOpenAI(model="deepseek-chat",
               base_url=os.getenv("DEEPSEEK_BASE_URL"),
               api_key=os.getenv("DEEPSEEK_API_KEY"),
               temperature=0)

In [ ]:
# 打开文件，并赋予读取模式
with open("./company.txt",'r',encoding='utf-8') as file:
    content=file.read()
    print(content)

In [ ]:
from langchain_core.documents import Document

documents=[Document(page_content=content)]

In [ ]:
documents

In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

print(os.getenv("NEO4J_URI"))
print(os.getenv("NEO4J_USERNAME"))
print(os.getenv("NEO4J_PASSWORD"))
print(os.getenv("NEO4J_DATABASE"))





In [ ]:
graph=Neo4jGraph(url=os.getenv("NEO4J_URI"),
                 username=os.getenv("NEO4J_USERNAME"),
                 password=os.getenv("NEO4J_PASSWORD"),
                 database=os.getenv("NEO4J_DATABASE"))

In [ ]:
# 图转换器配置
graph_transformer=LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["公司","产品","技术","市场","活动","合作伙伴"],
    allowed_relationships=["推出","参与","合作","位于","开发"]
)

graph_transformer=LLMGraphTransformer(llm=llm,ignore_tool_usage=True)

graph_documents=graph_transformer.convert_to_graph_documents(documents)

graph.add_graph_documents(graph_documents)


print(f"Graph documents: {len(graph_documents)}")
print(f"Nodes from 1st graph doc: {graph_documents[0].nodes}")
print(f"Relationships from 1st graph doc: {graph_documents[0].relationships}")

In [28]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

# 自定义 Cypher 生成 prompt - 加强约束
# --- 自定义 prompt：强调用精确 ID + CONTAINS 兜底 ---
CUSTOM_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""Task: Generate ONE Cypher query to answer the question about a Chinese company.

CRITICAL RULES:
1. Identify the entity keyword from the question (e.g. "小米", "苹果", "华为")
2. ALWAYS use CONTAINS for text matching — NEVER use = for Chinese company names
   Example: WHERE n.id CONTAINS '小米'  (NOT: WHERE n.id = '小米公司')
3. Match ONLY the keyword part (before 公司/科技/技术), because the full name in the database may differ
4. Return ONLY the Cypher statement. No markdown, no explanation, no ```cypher```.

Schema:
{schema}

Question: {question}

Cypher:"""
)

cypher_chain=GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=llm,
    cypher_prompt=CUSTOM_PROMPT,
    qa_llm=llm,
    validate_cypher=True,
    verbose=True,
    allow_dangerous_requests=True
)

cypher_chain.invoke("苹果公司开发了什么")





> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company)-[:DEVELOPS]->(t:Technology) WHERE c.id CONTAINS '苹果' RETURN t.id
Full Context:
[{'t.id': 'Siri语音助手'}, {'t.id': '面部识别技术'}]

> Finished chain.


{'query': '苹果公司开发了什么', 'result': '苹果公司开发了Siri语音助手和面部识别技术。'}

In [29]:
cypher_chain.invoke("都有哪些公司在数据库中")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company) RETURN c.id
Full Context:
[{'c.id': '谷歌'}, {'c.id': 'Adobe'}, {'c.id': '小米科技有限责任公司'}, {'c.id': '微软'}, {'c.id': '华为技术有限公司'}, {'c.id': '苹果公司'}, {'c.id': '英特尔'}]

> Finished chain.


{'query': '都有哪些公司在数据库中',
 'result': '在数据库中包含的公司有：谷歌、Adobe、小米科技有限责任公司、微软、华为技术有限公司、苹果公司以及英特尔。'}

In [25]:
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os; load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat",
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0
)

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE")
)

# --- 构建增强版 Schema：包含实际节点 ID ---
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)
with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
    result = session.run(
        "MATCH (n) RETURN DISTINCT labels(n)[0] as label, "
        "collect(DISTINCT n.id)[..15] as sample_ids"
    )
    enriched = graph.schema + "\n\n=== 实际节点ID（查询时必须用这些精确值）===\n"
    for r in result:
        enriched += f"{r['label']}: {r['sample_ids']}\n"
driver.close()

graph.schema = enriched  # 替换默认 schema

# --- 自定义 prompt：强调用精确 ID + CONTAINS 兜底 ---
CUSTOM_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""Task: Generate ONE Cypher query to answer the question about a Chinese company.

CRITICAL RULES:
1. Identify the entity keyword from the question (e.g. "小米", "苹果", "华为")
2. ALWAYS use CONTAINS for text matching — NEVER use = for Chinese company names
   Example: WHERE n.id CONTAINS '小米'  (NOT: WHERE n.id = '小米公司')
3. Match ONLY the keyword part (before 公司/科技/技术), because the full name in the database may differ
4. Return ONLY the Cypher statement. No markdown, no explanation, no ```cypher```.

Schema:
{schema}

Question: {question}

Cypher:"""
)


chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=CUSTOM_PROMPT,
    verbose=True,
    allow_dangerous_requests=True,
)

chain.invoke("介绍一下小米公司")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company) WHERE c.id CONTAINS '小米' RETURN c
Full Context:
[{'c': {'id': '小米科技有限责任公司'}}]

> Finished chain.


{'query': '介绍一下小米公司', 'result': '小米公司，全称为小米科技有限责任公司，是一家专注于智能硬件和电子产品研发的科技企业。'}

In [26]:
chain.invoke("介绍一下苹果公司")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company) WHERE c.id CONTAINS '苹果' RETURN c
Full Context:
[{'c': {'id': '苹果公司'}}]

> Finished chain.


{'query': '介绍一下苹果公司', 'result': '苹果公司是一家知名的科技企业，主要从事消费电子、软件和在线服务的研发与销售。'}

In [27]:
chain.invoke("苹果公司开发了什么")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company)-[:DEVELOPS]->(t:Technology) WHERE c.id CONTAINS '苹果' RETURN t.id
Full Context:
[{'t.id': 'Siri语音助手'}, {'t.id': '面部识别技术'}]

> Finished chain.


{'query': '苹果公司开发了什么', 'result': '苹果公司开发了Siri语音助手和面部识别技术。'}

In [ ]:
from langgraph.graph import StateGraph,MessagesState ,START,END

class AgentState(MessagesState):
    next:str

In [32]:
from langchain.messages import HumanMessage

def graph_kg(state:AgentState):
    messages=state["messages"][-1]
    
    # 自定义 Cypher 生成 prompt - 加强约束
    # --- 自定义 prompt：强调用精确 ID + CONTAINS 兜底 ---
    CUSTOM_PROMPT = PromptTemplate(
        input_variables=["schema", "question"],
        template="""Task: Generate ONE Cypher query to answer the question about a Chinese company.

    CRITICAL RULES:
    1. Identify the entity keyword from the question (e.g. "小米", "苹果", "华为")
    2. ALWAYS use CONTAINS for text matching — NEVER use = for Chinese company names
    Example: WHERE n.id CONTAINS '小米'  (NOT: WHERE n.id = '小米公司')
    3. Match ONLY the keyword part (before 公司/科技/技术), because the full name in the database may differ
    4. Return ONLY the Cypher statement. No markdown, no explanation, no ```cypher```.

    Schema:
    {schema}

    Question: {question}

    Cypher:"""
    )

    cypher_chain=GraphCypherQAChain.from_llm(
        graph=graph,
        cypher_llm=llm,
        cypher_prompt=CUSTOM_PROMPT,
        qa_llm=llm,
        validate_cypher=True,
        verbose=True,
        allow_dangerous_requests=True
    )
    response=cypher_chain.invoke(messages.content)
    final_response=[HumanMessage(content=response['result'],name="graph_kg")]
    return {"messages":final_response}